# 03_01 — обучаемый Word2Vec baseline

Копия эксперимента `03_WSD`, превращённая в воспроизводимый supervised baseline. Используются исходные НКРЯ Word2Vec-векторы, контекстное окно, POS и multilabel-классификатор.

Разделение данных и метрики полностью совпадают с `02_01`, поэтому результаты можно сравнивать напрямую.


#### Назначение следующей ячейки

Устанавливает библиотеки, без которых следующие ячейки не выполнятся. Ячейку достаточно запускать один раз после создания или сброса окружения.


In [7]:
# --- Пояснения и точки настройки ---
# Назначение: Устанавливает библиотеки, без которых следующие ячейки не выполнятся. Ячейку
# достаточно запускать один раз после создания или сброса окружения.
# Настройка: Можно зафиксировать версии пакетов (`package==version`) для воспроизводимости. После
# обновления PyTorch, Transformers или scikit-learn желательно перезапустить kernel.
# --- Конец пояснений ---

%pip install -q gensim scipy pandas scikit-learn tqdm


## 1. Нормализация CoNLL-U и единое оценивание

#### Назначение следующей ячейки

Содержит общий слой чтения CoNLL-U: нормализует разные варианты `sem=[...]`, читает предложения потоково, атомарно сохраняет кэш и рассчитывает единые multilabel-метрики.


In [8]:
# --- Пояснения и точки настройки ---
# Назначение: Содержит общий слой чтения CoNLL-U: нормализует разные варианты `sem=[...]`, читает
# предложения потоково, атомарно сохраняет кэш и рассчитывает единые multilabel-метрики.
# Настройка: При появлении нового формата MISC следует расширять `tags_from_misc`, сохраняя
# lowercase, удаление кавычек и дедупликацию. Пороговые метрики можно дополнить PR-AUC, но порядок
# `CLASSES` менять нельзя.
# --- Конец пояснений ---

import ast
import hashlib
import json
import os
import pickle
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEM_RE = re.compile(r"(?:^|\|)sem=(\[[^\]]*\]|[^|]+)")


def normalize_sem_tags(value):
    """Normalize sem=[a,b], sem=['a','b'] and sem=[\"a\", \"b\"]."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_tags = value
    else:
        text = str(value).strip()
        try:
            # Шаг: пробуем разобрать значение MISC как настоящий Python-список с кавычками.
            parsed = ast.literal_eval(text)
            raw_tags = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            raw_tags = text.strip("[]").split(",")
    return sorted({
        str(tag).strip().strip("'\"").lower()
        for tag in raw_tags
        if str(tag).strip().strip("'\"")
    })


def tags_from_misc(misc):
    # Шаг: извлекаем только значение поля sem, не затрагивая остальные атрибуты MISC.
    match = SEM_RE.search(str(misc or ""))
    if not match:
        return [], False
    return normalize_sem_tags(match.group(1)), True


@dataclass
class Sentence:
    words: list[str]
    lemmas: list[str]
    upos: list[str]
    tags: list[list[str]]
    labeled: list[bool]
    source: str


def read_conllu(path):
    """Stream CoNLL-U; skip multiword and empty-node rows."""
    sentences, rows = [], []
    with Path(path).open(encoding="utf-8") as file:
        for line in file:
            line = line.rstrip("\n")
            if not line:
                if rows:
                    # Шаг: завершаем накопленное предложение при встрече пустой строки CoNLL-U.
                    sentences.append(_rows_to_sentence(rows, str(path)))
                    rows = []
                continue
            if line.startswith("#"):
                continue
            columns = line.split("\t")
            if len(columns) == 10 and columns[0].isdigit():
                rows.append(columns)
    if rows:
        # Шаг: завершаем накопленное предложение при встрече пустой строки CoNLL-U.
        sentences.append(_rows_to_sentence(rows, str(path)))
    return sentences


def _rows_to_sentence(rows, source):
    tags_and_masks = [tags_from_misc(row[9]) for row in rows]
    return Sentence(
        words=[row[1] for row in rows],
        lemmas=[row[2] if row[2] != "_" else row[1].lower() for row in rows],
        upos=[row[3] for row in rows],
        tags=[item[0] for item in tags_and_masks],
        labeled=[item[1] for item in tags_and_masks],
        source=source,
    )


def file_signature(paths):
    digest = hashlib.sha256()
    for path in sorted(map(Path, paths)):
        stat = path.stat()
        digest.update(f"{path}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    return digest.hexdigest()[:16]


def atomic_pickle(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as file:
        # Шаг: сериализуем дорогой промежуточный результат для повторного использования.
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    # Шаг: атомарно заменяем checkpoint только после успешной полной записи.
    temporary.replace(path)


def load_records(paths, cache_dir, name):
    """Load and checkpoint every source file independently."""
    paths = list(map(Path, paths))
    file_cache_dir = Path(cache_dir) / "parsed_conllu"
    file_cache_dir.mkdir(parents=True, exist_ok=True)
    records = []
    progress = tqdm(paths, desc=f"Чтение {name}", unit="файл")
    for path in progress:
        cache = file_cache_dir / f"{path.stem}_{file_signature([path])}.pkl"
        file_records = None
        if cache.exists():
            try:
                with cache.open("rb") as file:
                    # Шаг: восстанавливаем уже разобранный файл, чтобы не повторять длительный парсинг.
                    file_records = pickle.load(file)
                status = "cache"
            except Exception:
                file_records = None
        if file_records is None:
            # Шаг: читаем новый файл и оставляем только предложения с sem-разметкой.
            file_records = [sentence for sentence in read_conllu(path)
                            if any(sentence.labeled)]
            atomic_pickle(file_records, cache)
            status = "saved"
        records.extend(file_records)
        progress.set_postfix(file=path.name[:24], status=status,
                             sentences=f"{len(records):,}")
    progress.close()
    print(f"{name}: {len(records):,} размеченных предложений")
    return records

def collect_classes(groups):
    # Шаг: строим единый детерминированный порядок классов для всех матриц и моделей.
    classes = sorted({
        tag
        for records in groups
        for sentence in records
        for tags, labeled in zip(sentence.tags, sentence.labeled)
        if labeled
        for tag in tags
    })
    if not classes:
        raise ValueError("Не найдено ни одной метки sem")
    return classes


def label_statistics(records, classes):
    counts = Counter(
        tag for sentence in records
        for tags, labeled in zip(sentence.tags, sentence.labeled)
        if labeled for tag in tags
    )
    return pd.DataFrame({"tag": classes, "support": [counts[tag] for tag in classes]}) \
        .sort_values("support", ascending=False).reset_index(drop=True)


def tune_thresholds(y_true, probabilities, min_support=2):
    from sklearn.metrics import precision_recall_curve
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    for column in range(y_true.shape[1]):
        if y_true[:, column].sum() < min_support:
            continue
        # Шаг: перебираем возможные пороги конкретного класса по validation-разметке.
        precision, recall, candidates = precision_recall_curve(
            y_true[:, column], probabilities[:, column]
        )
        if not len(candidates):
            continue
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(
            precision[:-1] + recall[:-1], 1e-12
        )
        thresholds[column] = candidates[int(np.nanargmax(f1))]
    return thresholds


def multilabel_metrics(y_true, probabilities, thresholds, classes, split):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score, hamming_loss
    # Шаг: превращаем вероятности в multilabel-предсказания с отдельным порогом каждого тега.
    predicted = (probabilities >= thresholds).astype(np.int8)
    micro = precision_recall_fscore_support(
        y_true, predicted, average="micro", zero_division=0
    )
    supported = y_true.sum(axis=0) > 0
    macro = precision_recall_fscore_support(
        y_true[:, supported], predicted[:, supported], average="macro", zero_division=0
    )
    per_class = precision_recall_fscore_support(
        y_true, predicted, average=None, zero_division=0
    )
    summary = {
        "split": split,
        "tokens": int(len(y_true)),
        "labels": int(y_true.sum()),
        "micro_precision": micro[0], "micro_recall": micro[1], "micro_f1": micro[2],
        "macro_precision_supported": macro[0],
        "macro_recall_supported": macro[1],
        "macro_f1_supported": macro[2],
        "exact_match": accuracy_score(y_true, predicted),
        "hamming_loss": hamming_loss(y_true, predicted),
    }
    details = pd.DataFrame({
        "tag": classes, "precision": per_class[0], "recall": per_class[1],
        "f1": per_class[2], "support": per_class[3], "threshold": thresholds,
    }).sort_values("support", ascending=False)
    return summary, details


## 2. Пути и разделение данных

#### Назначение следующей ячейки

Подключает Google Drive и импортирует общие зависимости. Диск используется для исходных корпусов, моделей и результатов, переживающих разрыв соединения с Colab.


In [9]:
# --- Пояснения и точки настройки ---
# Назначение: Подключает Google Drive и импортирует общие зависимости. Диск используется для
# исходных корпусов, моделей и результатов, переживающих разрыв соединения с Colab.
# Настройка: При локальном запуске блок Google Drive можно не использовать; достаточно указать
# существующий `PROJECT_ROOT`. Импорты удалять не следует, если соответствующие функции
# используются ниже.
# --- Конец пояснений ---

try:
    from google.colab import drive
    # Шаг: монтируем Google Drive; операция может потребовать подтверждения доступа.
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Аспирантура')
except ImportError:
    # Для локального запуска укажите корень репозитория.
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()

DATA_CANDIDATES = [PROJECT_ROOT / 'data/conllu', PROJECT_ROOT / 'conllu']
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
CACHE_ROOT = PROJECT_ROOT / 'outputs/experiment_cache'
LOCAL_CHECKPOINT_ROOT = Path(os.environ.get(
    'WSD_CHECKPOINT_DIR', CACHE_ROOT / 'local_checkpoints'
))
for directory in (CACHE_ROOT, LOCAL_CHECKPOINT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print('Локальные checkpoints:', LOCAL_CHECKPOINT_ROOT)
print('Каталог outputs/ исключён из Git и не будет загружен в репозиторий.')

# Шаг: собираем все автоматически размеченные корпуса, не включая ручные test-наборы.
TRAIN_FILES = sorted([
    *DATA_ROOT.glob('bot/*.conllu'),
    *DATA_ROOT.glob('cobald/*.conllu'),
    *DATA_ROOT.glob('syntagrus/*.conllu'),
])
HAND_ROOT = DATA_ROOT / 'handlabeled'
VALID_FILE = HAND_ROOT / 'GramEval2020-test-fiction.conllu'
TEST_FILES = {
    genre: HAND_ROOT / f'GramEval2020-test-{genre}.conllu'
    for genre in ('news', 'poetry', 'social', 'wiki')
}

assert TRAIN_FILES, f'Не найдены обучающие файлы в {DATA_ROOT}'
assert VALID_FILE.exists(), VALID_FILE
print(f'Обучающих файлов: {len(TRAIN_FILES)}')
print('Validation:', VALID_FILE.name)
print('Test:', ', '.join(path.name for path in TEST_FILES.values()))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Локальные checkpoints: /content/drive/MyDrive/Colab Notebooks/Аспирантура/outputs/experiment_cache/local_checkpoints
Каталог outputs/ исключён из Git и не будет загружен в репозиторий.
Обучающих файлов: 11
Validation: GramEval2020-test-fiction.conllu
Test: GramEval2020-test-news.conllu, GramEval2020-test-poetry.conllu, GramEval2020-test-social.conllu, GramEval2020-test-wiki.conllu


## 3. Загрузка корпусов и канонический набор классов

#### Назначение следующей ячейки

Загружает размеченные предложения, восстанавливая готовые файлы из кэша, и строит канонический набор классов по всем используемым меткам.


In [10]:
# --- Пояснения и точки настройки ---
# Назначение: Загружает размеченные предложения, восстанавливая готовые файлы из кэша, и строит
# канонический набор классов по всем используемым меткам. Тестовые тексты участвуют только в
# определении имён классов.
# Настройка: Можно очищать кэш при изменении парсера. Не задавайте классы вручную частичным
# списком: это приведёт к игнорированию неизвестных gold-меток.
# --- Конец пояснений ---

# Шаг: загружаем train с восстановлением готовых файлов из локального кэша.
train_records = load_records(TRAIN_FILES, CACHE_ROOT, 'train_all')
# Шаг: загружаем ручной validation-набор, используемый только для настройки порогов.
valid_records = load_records([VALID_FILE], CACHE_ROOT, 'valid_fiction')
# Шаг: создаём отдельные ручные тесты по жанрам без смешивания с обучением.
test_records = {
    genre: load_records([path], CACHE_ROOT, f'test_{genre}')
    for genre, path in TEST_FILES.items()
}

# Канонический набор строится по ВСЕМ используемым меткам, включая ручные тесты.
# Сами тестовые предложения при этом никогда не используются для обучения.
# Шаг: фиксируем полный канонический список меток до построения выходных слоёв.
CLASSES = collect_classes([
    train_records, valid_records, *test_records.values()
])
TAG_TO_ID = {tag: index for index, tag in enumerate(CLASSES)}
(CACHE_ROOT / 'canonical_classes.json').write_text(
    json.dumps(CLASSES, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Канонических классов: {len(CLASSES)}')
print(CLASSES)
display(label_statistics(train_records, CLASSES).head(30))


Чтение train_all:   0%|          | 0/11 [00:00<?, ?файл/s]

train_all: 103,704 размеченных предложений


Чтение valid_fiction:   0%|          | 0/1 [00:00<?, ?файл/s]

valid_fiction: 80 размеченных предложений


Чтение test_news:   0%|          | 0/1 [00:00<?, ?файл/s]

...

Чтение test_social:   0%|          | 0/1 [00:00<?, ?файл/s]

test_social: 109 размеченных предложений


Чтение test_wiki:   0%|          | 0/1 [00:00<?, ?файл/s]

test_wiki: 73 размеченных предложений
Канонических классов: 84
['abstr', 'age', 'animal', 'appear', 'art', 'artwork', 'be', 'behav', 'berry', 'changest', 'cin', 'class', 'cloth', 'color', 'concr', 'constr', 'contact', 'degr', 'degree', 'device', 'dir', 'disappear', 'disease', 'dish', 'dist', 'doc', 'dram', 'emot', 'etn', 'ev', 'famn', 'food', 'form', 'fruit', 'furn', 'grain', 'group', 'hum', 'humq', 'impact', 'instr', 'inter', 'kin', 'light', 'max', 'ment', 'min', 'move', 'mus', 'neg', 'org', 'part', 'patrn', 'perc', 'physiol', 'physq', 'place', 'plant', 'posit', 'poss', 'prof', 'propn', 'psych', 'put', 'qtm', 'quant', 'set', 'size', 'sound', 'speech', 'speed', 'sport', 'stuff', 'taste', 'temper', 'text', 'time', 'tool', 'topon', 'transp', 'unit', 'weapon', 'weather', 'weight']


,tag,support
0,abstr,78110
1,concr,76681
2,time,34832
3,hum,33579
4,org,25991
5,place,18336
6,ev,13626
7,move,13382
8,degr,11700
9,topon,11146


## 4. Word2Vec-признаки: слово, контекст и POS

#### Назначение следующей ячейки

Загружает модель НКРЯ и строит признаки размеченного слова: его Word2Vec-вектор, средний вектор локального контекста и one-hot часть речи.


In [11]:
# --- Пояснения и точки настройки ---
# Назначение: Загружает модель НКРЯ и строит признаки размеченного слова: его Word2Vec-вектор,
# средний вектор локального контекста и one-hot часть речи. Признаки сохраняются пакетами и
# восстанавливаются после перезапуска.
# Настройка: Можно менять размер `window` и `sentences_per_checkpoint`. При замене Word2Vec-модели
# следует очистить старый кэш признаков, поскольку размерность и словарь модели могли измениться.
# --- Конец пояснений ---

from gensim.models import Word2Vec
from scipy import sparse
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDClassifier

MODEL_PATH = PROJECT_ROOT / 'main_POS_V300_W5_C5.model'
# Шаг: загружаем модель НКРЯ; все последующие ключи должны иметь формат лемма_POS.
word2vec = Word2Vec.load(str(MODEL_PATH)).wv

POS_MAP = {
    'NOUN':'S', 'PROPN':'S', 'ADJ':'A', 'VERB':'V', 'AUX':'V',
    'ADV':'ADV', 'ADVPRO':'ADV', 'PRON':'SPRO', 'DET':'APRO',
    'NUM':'NUM', 'ADP':'PR', 'CCONJ':'CONJ', 'SCONJ':'CONJ',
    'PART':'PART', 'INTJ':'INTJ',
}
POS_CLASSES = sorted({pos for records in [train_records, valid_records, *test_records.values()]
                      for sentence in records for pos in sentence.upos})
POS_TO_ID = {pos: i for i, pos in enumerate(POS_CLASSES)}


def word_vector(lemma, upos):
    key = f"{lemma.lower()}_{POS_MAP.get(upos, upos)}"
    return word2vec[key] if key in word2vec else None


def build_features(records, classes, window=5, show_progress=True):
    tag_to_id = {tag: i for i, tag in enumerate(classes)}
    dimension = word2vec.vector_size
    features, targets = [], []
    iterator = tqdm(records, desc='Word2Vec-признаки', unit='предл.') if show_progress else records
    for sentence in iterator:
        # Шаг: заранее извлекаем векторы слов предложения, сохраняя None для OOV.
        vectors = [word_vector(lemma, pos)
                   for lemma, pos in zip(sentence.lemmas, sentence.upos)]
        for index, (tags, labeled) in enumerate(zip(sentence.tags, sentence.labeled)):
            if not labeled:
                continue
            target = vectors[index]
            # Шаг: собираем известные Word2Vec-векторы соседей, исключая целевое слово.
            context = [vectors[j] for j in range(max(0,index-window),
                       min(len(vectors),index+window+1))
                       if j != index and vectors[j] is not None]
            target_vec = target if target is not None else np.zeros(dimension)
            context_vec = np.mean(context, axis=0) if context else np.zeros(dimension)
            pos_vec = np.zeros(len(POS_CLASSES), dtype=np.float32)
            pos_vec[POS_TO_ID[sentence.upos[index]]] = 1.0
            # Шаг: объединяем целевое слово, контекст и POS в один обучающий вектор.
            features.append(np.concatenate([target_vec, context_vec, pos_vec]))
            row = np.zeros(len(classes), dtype=np.int8)
            for tag in tags:
                row[tag_to_id[tag]] = 1
            targets.append(row)
    return np.asarray(features, dtype=np.float32), np.asarray(targets, dtype=np.int8)


def cached_features(records, name, sentences_per_checkpoint=2000):
    class_signature = hashlib.sha256('\0'.join(CLASSES).encode()).hexdigest()[:12]
    sources = sorted({sentence.source for sentence in records})
    source_signature = file_signature(sources)
    job_dir = LOCAL_CHECKPOINT_ROOT / '03_01_features' / (
        f'{name}_{class_signature}_{source_signature}'
    )
    job_dir.mkdir(parents=True, exist_ok=True)
    x_parts, y_parts = [], []
    progress = tqdm(total=len(records), desc=f'Признаки {name}', unit='предл.')
    for start in range(0, len(records), sentences_per_checkpoint):
        end = min(start + sentences_per_checkpoint, len(records))
        checkpoint = job_dir / f'{start:07d}_{end:07d}.npz'
        loaded = False
        if checkpoint.exists():
            try:
                with np.load(checkpoint) as saved:
                    x_part, y_part = saved['x'], saved['y']
                loaded = True
            except Exception:
                loaded = False
        if not loaded:
            x_part, y_part = build_features(records[start:end], CLASSES,
                                            show_progress=False)
            temporary = checkpoint.with_suffix('.tmp.npz')
            # Шаг: сохраняем очередную часть признаков; готовые части будут пропущены при перезапуске.
            np.savez_compressed(temporary, x=x_part, y=y_part)
            temporary.replace(checkpoint)
        x_parts.append(x_part)
        y_parts.append(y_part)
        progress.update(end - start)
        progress.set_postfix(status='cache' if loaded else 'saved',
                             checkpoint=checkpoint.name)
    progress.close()
    return np.concatenate(x_parts), np.concatenate(y_parts)

x_train, y_train = cached_features(train_records, 'train')
x_valid, y_valid = cached_features(valid_records, 'valid_fiction')
print('Train:', x_train.shape, y_train.shape)


Признаки train:   0%|          | 0/103704 [00:00<?, ?предл./s]

Признаки valid_fiction:   0%|          | 0/80 [00:00<?, ?предл./s]

Train: (307204, 628) (307204, 84)


## 5. Обучение One-vs-Rest классификатора

#### Назначение следующей ячейки

Обучает отдельный логистический классификатор для каждого тега. После каждого класса список моделей сохраняется атомарно; редкие или константные классы обрабатываются `DummyClassifier`.


In [12]:
# --- Пояснения и точки настройки ---
# Назначение: Обучает отдельный логистический классификатор для каждого тега. После каждого класса
# список моделей сохраняется атомарно; редкие или константные классы обрабатываются
# `DummyClassifier`.
# Настройка: Можно настраивать `alpha`, `l1_ratio`, `max_iter` и `tol`. `early_stopping` оставлен
# выключенным, так как внутренний stratified split падает на классах с одним положительным
# примером.
# --- Конец пояснений ---

from sklearn.dummy import DummyClassifier

RUN_CHECKPOINT_DIR = LOCAL_CHECKPOINT_ROOT / '03_01_word2vec'
RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = RUN_CHECKPOINT_DIR / 'ovr_by_class.pkl'


class CheckpointedOVR:
    def __init__(self, estimators):
        self.estimators_ = estimators

    def predict_proba(self, x):
        columns = []
        for estimator in tqdm(self.estimators_, desc='Вероятности', unit='класс'):
            proba = estimator.predict_proba(x)
            # DummyClassifier for a constant class may expose a single column.
            if proba.shape[1] == 1:
                positive = np.full(len(x), float(estimator.classes_[0] == 1))
            else:
                positive_index = int(np.where(estimator.classes_ == 1)[0][0])
                positive = proba[:, positive_index]
            columns.append(positive)
        return np.column_stack(columns)


estimators = [None] * len(CLASSES)
if MODEL_FILE.exists():
    try:
        with MODEL_FILE.open('rb') as file:
            # Шаг: восстанавливаем ранее рассчитанный объект из доверенного checkpoint.
            saved_model = pickle.load(file)
        if saved_model.get('classes') == CLASSES:
            estimators = saved_model['estimators']
            print(f"Продолжение: готово {sum(e is not None for e in estimators)}/{len(CLASSES)} классов")
    except Exception as error:
        print('Checkpoint не прочитан, обучение начнётся заново:', error)

progress = tqdm(total=len(CLASSES),
                initial=sum(e is not None for e in estimators),
                desc='Обучение OVR', unit='класс')
for column, tag in enumerate(CLASSES):
    if estimators[column] is not None:
        continue
    target = y_train[:, column]
    # Шаг: обрабатываем класс без вариативности константной моделью вместо SGD.
    if np.unique(target).size == 1:
        estimator = DummyClassifier(strategy='constant', constant=int(target[0]))
    else:
        # Шаг: создаём независимый бинарный классификатор текущего семантического тега.
        estimator = SGDClassifier(
            loss='log_loss', penalty='elasticnet', alpha=1e-5,
            l1_ratio=0.05, class_weight='balanced', max_iter=50,
            early_stopping=False, tol=1e-3,
            random_state=42 + column, n_jobs=1,
        )
    # Шаг: обучаем один OVR-классификатор на общем наборе признаков.
    estimator.fit(x_train, target)
    estimators[column] = estimator
    atomic_pickle({'estimators': estimators, 'classes': CLASSES}, MODEL_FILE)
    progress.update(1)
    progress.set_postfix(tag=tag, positives=int(target.sum()), checkpoint='saved')
progress.close()

classifier = CheckpointedOVR(estimators)
# Шаг: получаем вероятности всех тегов для последующей пороговой калибровки.
p_valid = classifier.predict_proba(x_valid)
THRESHOLDS = tune_thresholds(y_valid, p_valid)
np.save(RUN_CHECKPOINT_DIR / 'thresholds.npy', THRESHOLDS)


Обучение OVR:   0%|          | 0/84 [00:00<?, ?класс/s]

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_stochastic

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

## 6. Per-class thresholds и жанровая проверка

#### Назначение следующей ячейки

Вычисляет вероятности Word2Vec-классификатора, применяет пороги, настроенные только на validation, и формирует сопоставимые отчёты для всех ручных жанров.


In [ ]:
# --- Пояснения и точки настройки ---
# Назначение: Вычисляет вероятности Word2Vec-классификатора, применяет пороги, настроенные только
# на validation, и формирует сопоставимые отчёты для всех ручных жанров.
# Настройка: Можно добавлять новые отчёты и сохраняемые форматы. Для честного сравнения с RuBERT
# сохраняйте тот же список классов, разбиение корпусов и набор метрик.
# --- Конец пояснений ---

summaries, detail_tables = [], {}
summary, details = multilabel_metrics(
    y_valid, p_valid, THRESHOLDS, CLASSES, 'validation_fiction'
)
summaries.append(summary)
detail_tables['validation_fiction'] = details

for genre, records in test_records.items():
    x_test, y_test = cached_features(records, f'test_{genre}')
    # Шаг: получаем вероятности всех тегов для последующей пороговой калибровки.
    probabilities = classifier.predict_proba(x_test)
    summary, details = multilabel_metrics(
        y_test, probabilities, THRESHOLDS, CLASSES, f'test_{genre}'
    )
    summaries.append(summary)
    detail_tables[genre] = details

summary_table = pd.DataFrame(summaries)
display(summary_table)
display(detail_tables['validation_fiction'].head(40))
summary_table.to_csv(CACHE_ROOT / '03_01_metrics.csv', index=False)


Признаки test_news:   0%|          | 0/86 [00:00<?, ?предл./s]

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

Признаки test_poetry:   0%|          | 0/77 [00:00<?, ?предл./s]

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

Признаки test_social:   0%|          | 0/109 [00:00<?, ?предл./s]

...

Признаки test_wiki:   0%|          | 0/73 [00:00<?, ?предл./s]

Вероятности:   0%|          | 0/84 [00:00<?, ?класс/s]

,split,tokens,labels,micro_precision,micro_recall,micro_f1,macro_precision_supported,macro_recall_supported,macro_f1_supported,exact_match,hamming_loss
0,validation_fiction,512,949,0.133835,0.806112,0.229557,0.333367,0.811212,0.400913,0.0,0.119396
1,test_news,583,1144,0.127128,0.685315,0.214471,0.357635,0.666616,0.387841,0.0,0.117271
2,test_poetry,490,892,0.115821,0.744395,0.200453,0.261718,0.699368,0.315228,0.0,0.128693
3,test_social,455,842,0.114530,0.716152,0.197478,0.333474,0.765138,0.368111,0.0,0.128231
4,test_wiki,673,1213,0.110597,0.670239,0.189865,0.239980,0.535606,0.270908,0.0,0.122727


,tag,precision,recall,f1,support,threshold
14,concr,0.756637,0.919355,0.830097,186,8.903351e-02
37,hum,0.414773,0.929936,0.573674,157,2.374122e-10
0,abstr,0.761194,0.645570,0.698630,79,3.007706e-01
76,time,0.939394,0.794872,0.861111,39,9.622294e-01
51,part,0.642857,0.600000,0.620690,30,9.987617e-01
56,place,0.414634,0.566667,0.478873,30,1.692649e-01
47,move,0.941176,0.551724,0.695652,29,1.000000e+00
61,propn,0.384615,0.740741,0.506329,27,1.984504e-01
69,speech,0.827586,0.960000,0.888889,25,1.000000e+00
27,emot,0.044922,1.000000,0.085981,23,0.000000e+00


: 